# Adding Conversions to the Energy System Model

In the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb), we initialized an energy system model, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **Conversions components**. Conversions components represent components that **converts one or several commodities into other commodities of the Energy System  Model**. They can be seen as black boxes, using the inputs commodities to create the output commodities. We focus in this notebook on the most essential parameters required to define and understand a Conversion component, while more advanced and optional settings will be explained in subsequent notebooks.

Typical examples of sources include:

- a co-generation power plant using natural gas to produce electricity and heat 
- an electrolyzer using electricity to produce hydrogen
- a chemical plant using hydrogen, CO2, heat and electricity to produce methanol


## Initialize an energy system model

Before we can add sources, we need to initialize the energy system model as shown in the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb).

In [ ]:
import fine as fn  # Provides objects and functions to model an energy system
import pandas as pd  # Used to manage data in tables
import numpy as np  # Used to generate random input data
np.random.seed(42)  # Sets a "seed" to produce the same random input data in each model run

esM = fn.EnergySystemModel(
    locations = {"regionN", "regionS"},
    commodities = {"electricity", "naturalGas", "CO2"},
    commodityUnitsDict = {
    "electricity": r"GW$_{el}$",
    "naturalGas": r"GW$_{CH_{4},LHV}$",
    "CO2": r"Mio. t$_{CO_2}$/h",
    },
    costUnit = "1e6 Euro",
    lengthUnit = "km",
    numberOfTimeSteps = 8760,
    hoursPerTimeStep = 1
)

## Add Conversions

### Combined Cycle gas turbine plant

We can now add combined cycle gas turbine plant as a conversion component. Below you can find a more detailed explanation of the parameters used here.

In [ ]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="CCGT plants (methane)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={
            "electricity": 1,
            "methane": -1 / 0.6,
            "CO2": 201 * 1e-6 / 0.6,
        },
        hasCapacityVariable=True,
        investPerCapacity=0.65,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

## General Structure of a Conversion Instance

The following code snippet shows the structure of the `Conversion` class and its arguments.

```python
Conversion(
    esM,                                    # defined in this notebook
    name,                                   # defined in this notebook
    physicalUnit,                           # defined in this notebook
    commodityConversionFactors,             # defined in this notebook
    hasCapacityVariable=True,               # defined in this notebook
    capacityVariableDomain="continuous",
    capacityPerPlantUnit=1,
    linkedConversionCapacityID=None,
    hasIsBuiltBinaryVariable=False,
    bigM=None,
    operationRateMin=None,                  # defined in this notebook
    operationRateMax=None,                  # defined in this notebook
    operationRateFix=None,                  # defined in this notebook
    tsaWeight=1,
    locationalEligibility=None,
    capacityMin=None,
    capacityMax=None,
    partLoadMin=None,
    sharedPotentialID=None,
    linkedQuantityID=None,
    capacityFix=None,
    commissioningMin=None,
    commissioningMax=None,
    commissioningFix=None,
    isBuiltFix=None,
    investPerCapacity=0,
    investIfBuilt=0,
    opexPerOperation=0,
    opexPerCapacity=0,
    opexIfBuilt=0,
    QPcostScale=0,
    interestRate=0.08,
    economicLifetime=10,
    technicalLifetime=None,
    yearlyFullLoadHoursMin=None,
    yearlyFullLoadHoursMax=None,
    stockCommissioning=None,
    floorTechnicalLifetime=True,
    commissioningDependentCcf=False,
    emissionFactors=None,
    flowShares=None,
    pwlcfParameters=None,
    rampUpMax=None,
    rampDownMax=None,
    useTemporalCyclicConstraints=True,
)
```
In the following sections, we explain the most important arguments of a Conversion component.


## Required Arguments

### esM

`esM` is the energy system model to which the conversion is added.

### name

`name` is a string, which should describe the type of conversion which is added to the energy system model.

Examples:
- "nat_gas_power_plant"
- "PEM_electrolyzer"

### physicalUnit

`physicalUnit` defines the reference physical unit of the conversion component, to which maximum capacity limitations, cost parameters and the operation time series are all expressed.

Examples: 
- if `physicalUnit` = MW_{H2} for an electrolyzer, it means that its `capacityMax` of 10 MW is referred to hydrogen capacity and not electricity. 

### commodityConversionFactors

`commodityconversionfactor` specifies the conversion factors with which commodities are converted into each other with one unit of operation. The unit of operation were defined ealier in the [EnergySystemModel Initialization](../_01_initialize/_1_initialize_ESM.ipynb). The conversion factor related to this commodity is given as a float (constant), pandas.Series or pandas.DataFrame (time-variable). A negative value indicates that the commodity is consumed. A positive value indicates that the commodity is produced. Check unit consistency when specifying this parameter!

Examples: 
An electrolyzer converts, simply put, electricity into hydrogen with an electrical efficiency of 70%. The physicalUnit is given as GW_electric, the unit for the 'electricity' commodity is given in GW_electric and the 'hydrogen' commodity is given in GW_hydrogen_lowerHeatingValue -> the commodityConversionFactors are defined as {'electricity':-1,'hydrogen':0.7}.


### hasCapacityVariable

`hasCapacityVariable` is a **boolean**, which specifies whether the component has a capacity limit.

Examples:<br>
- A wind turbine has a capacity given in GW_electric -> ```hasCapacityVariable = True```
- Emitting CO2 into the environment is not per se limited by a capacity -> ```hasCapacityVariable = False```

## Optional Parameters

### operationRateMax

`operationRateMax` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### capacityMax 
`capacityMax` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### investPerCapacity

`investPerCapacity` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### opexPerCapacity

`opexPerCapacity` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)


### interestRate

`interestRate` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### economicLifetime

`economicLifetime` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

Many parameters were left out here. Some of them might need a page on their own. Others could be collected in an "other features" notebook